##  CoLA: a grammar-judgement dataset

**CoLA** (Corpus of Linguistic Acceptability, part of the GLUE benchmark) contains short sentences labelled by linguists as grammatically **acceptable (1)** or **unacceptable (0)**.

Plan: fine-tune a small transformer on CoLA to judge grammar, then score every sentence of every essay and turn those scores into essay-level grammar features (average acceptability, share of unacceptable sentences, worst sentence).

First step: load the data and look at it before training anything.

In [1]:
import os
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_HUB_DISABLE_XET"] = "1"
import torch, transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
print("transformers", transformers.__version__, "| CUDA:", torch.cuda.is_available())

path = snapshot_download("microsoft/deberta-v3-base", allow_patterns=["*.json", "spm.model", "pytorch_model.bin"])
tok = AutoTokenizer.from_pretrained(path)
print("tokenizer ids -> pad", tok.pad_token_id, "| cls", tok.cls_token_id, "| sep", tok.sep_token_id)

try:
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor(model_file=f"{path}/spm.model")
    print("SentencePiece reference:", {t: sp.piece_to_id(t) for t in ["[PAD]", "[CLS]", "[SEP]", "[UNK]"]})
except Exception as e:
    print("SentencePiece reference unavailable:", repr(e)[:100])

s = "he treats john very kind ."
print("\n1) default call :", tok.convert_ids_to_tokens(tok(s)["input_ids"]))
try:
    t2 = AutoTokenizer.from_pretrained(path)
    t2.add_bos_token = True; t2.add_eos_token = True
    print("2) flags fix    :", t2.convert_ids_to_tokens(t2(s)["input_ids"]))
except Exception as e:
    print("2) flags fix failed:", repr(e)[:150])

CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
def encode(text, max_len):
    ids = tok(text, add_special_tokens=False, truncation=True, max_length=max_len - 2)["input_ids"]
    return [CLS] + ids + [SEP]
print("3) manual wrap  :", tok.convert_ids_to_tokens(encode(s, 64)))
print("\nExpected ids: pad 0, cls 1, sep 2 ->", "OK" if (PAD, CLS, SEP) == (0, 1, 2) else "MISMATCH, stop and paste this")

transformers 5.0.0 | CUDA: True


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The tokenizer you are loading from '/root/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


tokenizer ids -> pad 0 | cls 1 | sep 2
SentencePiece reference: {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[UNK]': 3}

1) default call : ['▁he', '▁treats', '▁john', '▁very', '▁kind', '▁.']


The tokenizer you are loading from '/root/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


2) flags fix    : ['[CLS]', '▁he', '▁treats', '▁john', '▁very', '▁kind', '▁.', '[SEP]']
3) manual wrap  : ['[CLS]', '▁he', '▁treats', '▁john', '▁very', '▁kind', '▁.', '[SEP]']

Expected ids: pad 0, cls 1, sep 2 -> OK


In [2]:
import io, glob, time, random, zipfile, urllib.request
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import matthews_corrcoef, accuracy_score

if not glob.glob("/kaggle/working/cola/**/in_domain_train.tsv", recursive=True):
    raw = urllib.request.urlopen("https://nyu-mll.github.io/CoLA/cola_public_1.1.zip", timeout=30).read()
    zipfile.ZipFile(io.BytesIO(raw)).extractall("/kaggle/working/cola")
find = lambda n: glob.glob(f"/kaggle/working/cola/**/{n}", recursive=True)[0]
cols = ["source", "label", "note", "sentence"]
train = pd.read_csv(find("in_domain_train.tsv"), sep="\t", header=None, names=cols)
val   = pd.read_csv(find("in_domain_dev.tsv"),   sep="\t", header=None, names=cols)

SEED, EPOCHS, BATCH, LR, MAXLEN = 42, 4, 32, 2e-5, 64
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda:0")

def make_items(df):
    return [{"ids": encode(t, MAXLEN), "label": int(l)} for t, l in zip(df["sentence"], df["label"])]

def collate(batch):
    L = max(len(b["ids"]) for b in batch)
    ids = torch.full((len(batch), L), PAD, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["ids"]); ids[i, :n] = torch.tensor(b["ids"]); att[i, :n] = 1
    return {"input_ids": ids, "attention_mask": att, "labels": torch.tensor([b["label"] for b in batch])}

tr_items, va_items = make_items(train), make_items(val)
print("First training item:", tok.convert_ids_to_tokens(tr_items[0]["ids"]))
train_dl = DataLoader(tr_items, batch_size=BATCH, shuffle=True, collate_fn=collate)
val_dl   = DataLoader(va_items, batch_size=128, shuffle=False, collate_fn=collate)

model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=2).float().to(device)
no_decay = ["bias", "LayerNorm.weight"]
groups = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], "weight_decay": 0.01},
          {"params": [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], "weight_decay": 0.0}]
opt   = torch.optim.AdamW(groups, lr=LR)
total = EPOCHS * len(train_dl)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

def evaluate():
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for b in val_dl:
            b = {k: v.to(device) for k, v in b.items()}
            out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            preds += out.logits.argmax(-1).cpu().tolist(); labels += b["labels"].cpu().tolist()
    return matthews_corrcoef(labels, preds), accuracy_score(labels, preds), float(np.mean(preds))

best, t0, step = -1, time.time(), 0
for epoch in range(1, EPOCHS + 1):
    model.train(); losses, recent = [], []
    for b in train_dl:
        b = {k: v.to(device) for k, v in b.items()}
        loss = model(**b).loss
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        losses.append(loss.item()); recent.append(loss.item()); step += 1
        if step % 50 == 0:
            print(f"  step {step}: loss {np.mean(recent):.3f}"); recent = []
    mcc, acc, share1 = evaluate()
    print(f"epoch {epoch}: train loss {np.mean(losses):.3f} | val MCC {mcc:.3f} | accuracy {acc:.3f} "
          f"| predicted acceptable {share1 * 100:.0f}% | {(time.time() - t0) / 60:.1f} min")
    if mcc > best:
        best = mcc
        torch.save({k: v.half() for k, v in model.state_dict().items()}, "/kaggle/working/cola_deberta_best.pt")
print(f"\nBest validation MCC: {best:.3f}")

First training item: ['[CLS]', '▁Our', '▁friends', '▁won', "'", 't', '▁buy', '▁this', '▁analysis', ',', '▁let', '▁alone', '▁the', '▁next', '▁one', '▁we', '▁propose', '.', '[SEP]']


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSIN

  step 50: loss 0.662
  step 100: loss 0.497
  step 150: loss 0.407
  step 200: loss 0.347
  step 250: loss 0.342
epoch 1: train loss 0.443 | val MCC 0.707 | accuracy 0.879 | predicted acceptable 74% | 1.2 min
  step 300: loss 0.231
  step 350: loss 0.211
  step 400: loss 0.247
  step 450: loss 0.221
  step 500: loss 0.237
epoch 2: train loss 0.233 | val MCC 0.721 | accuracy 0.884 | predicted acceptable 77% | 2.5 min
  step 550: loss 0.130
  step 600: loss 0.135
  step 650: loss 0.131
  step 700: loss 0.145
  step 750: loss 0.156
  step 800: loss 0.140
epoch 3: train loss 0.140 | val MCC 0.702 | accuracy 0.877 | predicted acceptable 74% | 3.8 min
  step 850: loss 0.091
  step 900: loss 0.116
  step 950: loss 0.089
  step 1000: loss 0.078
  step 1050: loss 0.103
epoch 4: train loss 0.095 | val MCC 0.716 | accuracy 0.882 | predicted acceptable 74% | 5.1 min

Best validation MCC: 0.721


In [3]:
import re, glob, os, time

state = torch.load("/kaggle/working/cola_deberta_best.pt", map_location="cpu")
model.load_state_dict(state); model.eval()

root = os.path.dirname(glob.glob("/kaggle/input/**/ell_prepared.csv", recursive=True)[0])
ell = pd.read_csv(os.path.join(root, "ell_prepared.csv"))
trait_cols = ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]

SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")
def split_sentences(text):
    return [s.strip() for s in SPLIT.split(text) if len(s.split()) >= 3]

@torch.no_grad()
def score_sentences(sentences, batch=256):
    order = np.argsort([len(s) for s in sentences])
    probs = np.zeros(len(sentences), dtype=np.float32)
    for i in range(0, len(order), batch):
        idx = order[i:i + batch]
        b = collate([{"ids": encode(sentences[j].lower(), MAXLEN), "label": 0} for j in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits
        probs[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return probs

def cola_features(df):
    sents, owner = [], []
    for k, text in enumerate(df["text"]):
        ss = split_sentences(text); sents += ss; owner += [k] * len(ss)
    g = pd.DataFrame({"k": owner, "p": score_sentences(sents)}).groupby("k")["p"]
    out = pd.DataFrame({"cola_mean": g.mean(), "cola_min": g.min(), "cola_p25": g.quantile(0.25),
                        "cola_bad_share": g.apply(lambda x: (x < 0.5).mean()), "cola_n_sent": g.size()})
    out = out.reindex(range(len(df)))
    return out.fillna(out.median())

t0 = time.time()
ell_cola = cola_features(ell)
print(f"ELL scored in {(time.time() - t0) / 60:.1f} min")
ell_cola.assign(text_id=ell["text_id"].values).to_csv("/kaggle/working/ell_cola.csv", index=False)

print("\nSpearman correlation with the ELL scores:")
res = pd.DataFrame({t: ell_cola.corrwith(ell[t], method="spearman") for t in trait_cols})
res["average"] = ell_cola.corrwith(ell[trait_cols].mean(axis=1), method="spearman")
print(res.round(2).to_string())
print("\nFor comparison, LanguageTool grammar rate vs ELL grammar: -0.52")

ELL scored in 1.3 min

Spearman correlation with the ELL scores:
                cohesion  syntax  vocabulary  phraseology  grammar  conventions  average
cola_mean           0.42    0.53        0.48         0.57     0.67         0.47     0.62
cola_min            0.13    0.18        0.18         0.27     0.34         0.16     0.24
cola_p25            0.38    0.47        0.44         0.53     0.62         0.42     0.56
cola_bad_share     -0.36   -0.44       -0.42        -0.51    -0.60        -0.40    -0.54
cola_n_sent         0.35    0.37        0.36         0.31     0.24         0.32     0.38

For comparison, LanguageTool grammar rate vs ELL grammar: -0.52


In [4]:
aes  = pd.read_csv(os.path.join(root, "aes_prepared.csv"))
asap = pd.read_csv(os.path.join(root, "asap_prepared.csv"))

t0 = time.time()
aes_cola = cola_features(aes)
print(f"AES scored in {(time.time() - t0) / 60:.1f} min")
aes_cola.assign(essay_id=aes["essay_id"].values).to_csv("/kaggle/working/aes_cola.csv", index=False)

t0 = time.time()
asap_cola = cola_features(asap)
print(f"ASAP scored in {(time.time() - t0) / 60:.1f} min")
asap_cola.assign(essay_id=asap["essay_id"].values).to_csv("/kaggle/working/asap_cola.csv", index=False)

print("\nAES: Spearman correlation with the score")
print(aes_cola.corrwith(aes["score"], method="spearman").round(2).to_string())

print("\nASAP: cola_mean vs rescaled score, per essay set")
tmp = asap_cola.assign(score=asap["score_scaled"].values, essay_set=asap["essay_set"].values)
print(tmp.groupby("essay_set")[["cola_mean", "score"]]
         .apply(lambda g: g["cola_mean"].corr(g["score"], method="spearman")).round(2).to_string())

AES scored in 5.6 min
ASAP scored in 2.6 min

AES: Spearman correlation with the score
cola_mean         0.31
cola_min          0.07
cola_p25          0.29
cola_bad_share   -0.24
cola_n_sent       0.59

ASAP: cola_mean vs rescaled score, per essay set
essay_set
1    0.15
2    0.23
3    0.12
4    0.14
5    0.16
6    0.29
7    0.13
8    0.19
